# EXP-003-512 — M2 GraphCodeBERT trên Kaggle GPU

**Cùng recipe với EXP-002-512 (DEC-016)**: chỉ khác `model.hf_id`
(`microsoft/graphcodebert-base` thay cho `microsoft/codebert-base`). Cùng
max_length 512 (warm-start tu checkpoint 256), cùng batch/accum/lr/epochs, cùng runner
(`transformer_pair_runner`), cùng protocol chọn threshold trên validation.
Mọi khác biệt giữa M1 và M2 vì thế **chỉ** đến từ pretrained weights — đó là
điều kiện để so sánh ở RQ1 có ý nghĩa.

**Setup 1 lần trên Kaggle:** Settings → Accelerator = **GPU**, Internet = **ON**.

**Sửa đúng 1 chỗ:** cell 2 — dòng `REPO = "<user>/<repo>"`.
Chạy cell theo thứ tự. **Đừng bấm Run All** — cell 7 (train) mất ~7,8h.

### Ước lượng thời gian
- Train: **~7,8h** (đo từ throughput EXP-002-512: 8.400s/epoch × 3).
- Budget 9h dừng graceful trước giới hạn 12h của Kaggle.
- Evaluate: **~5 phút, chạy ngay trên Kaggle** (cell 8) → tải về chỉ vài MB.

### Nếu muốn train chung phiên với EXP-004
EXP-004 (M3 ASTNN) chỉ tốn ~35 phút nên chạy **trước** (DEC-023): nếu phiên này chết
giữa chừng thì bạn vẫn còn kết quả M3. Tổng ~8,4h < giới hạn 12h, vẫn trong một
ngày quota 30h GPU.

### Nếu session chết
1. Cài dataset cũ làm Input của notebook mới, chạy lại cell 2–4.
2. Chép `results/exp003_graphcodebert_poj104_512/last_checkpoint.pt` từ version đã
   save (file này **không** nằm trong tar cuối cùng — xem cell 9).
3. Cell 7 với `MODE = "resume"`: train tiếp epoch còn lại, không chạy lại từ đầu.

### Cảnh báo im lặng (đã từng xảy ra)
`python -m` mà **không in gì** nghĩa là module thiếu `if __name__ == "__main__"`
hoặc crash lúc import — dừng ngay, đừng đợi. Luôn chạy cell 5/6 (preflight,
diagnostic) trước cell 7: chúng in báo cáo và tốn <2 phút.

In [ ]:
import json
import os
import subprocess
import sys

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → thêm GITHUB_TOKEN, HF_TOKEN, attach vào notebook
# Cách 2: dán trực tiếp dưới đây (ĐỪNG share notebook này)
GITHUB_TOKEN = ""  # GitHub PAT — bắt buộc nếu repo PRIVATE
HF_TOKEN = ""  # HuggingFace token — tải checkpoint ổn định
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as e:
    print("dùng token dán tay:", e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("GITHUB_TOKEN:", "OK" if GITHUB_TOKEN else "TRỐNG", "| HF_TOKEN:", "OK" if HF_TOKEN else "TRỐNG")


In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sửa REPO trước khi chạy cell tiếp theo"

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lệnh thất bại (rc={r.returncode}): {cmd}")
    return r.stdout

def py(code, **kw):
    """Chạy snippet Python một dòng (không dùng newline trong code)."""
    return sh(f'{sys.executable} -c {json.dumps(code)}', **kw)

# Idempotent: chạy lại cell này sẽ pull bản mới thay vì clone lần nữa.
if os.path.isdir("project"):
    print("project đã tồn tại → pull thay vì clone")
    sh("git -C project pull --ff-only")
else:
    url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN else f"https://github.com/{REPO}.git"
    sh(f'git clone "{url}" project')
sh(f"git -C project remote set-url origin https://github.com/{REPO}.git")  # xóa token khỏi remote
sh("git -C project log --oneline -1")
os.chdir("project")
print("cwd =", os.getcwd())
leak = GITHUB_TOKEN and GITHUB_TOKEN in sh("git remote -v")
print("còn sót token trong remote?", "CÓ — xóa notebook!" if leak else "KHÔNG")

# GraphCodeBERT chỉ khác EXP-002-512 ở pretrained weights — assert để chắc chắn
from src.experiments import exp002_codebert as e2, exp003_graphcodebert as e3
assert set(e2.__all__) == set(e3.__all__), "wrapper lệch public API"
print("EXP-002-512/EXP-003-512 dùng CHUNG runner:", e2.train_model is e3.train_model)


In [ ]:
# KHÔNG đụng torch: Kaggle đã có sẵn bản CUDA tương thích, `pip install
# transformers` vô tình kéo torch về bản CPU/default sẽ làm train rất chậm (hoặc chết).
import importlib

need = []
for mod, pkg in [("transformers", "transformers"), ("datasets", "datasets"),
                 ("sklearn", "scikit-learn"), ("yaml", "pyyaml"),
                 ("tqdm", "tqdm"), ("numpy", "numpy"),
                 ("tree_sitter_cpp", "tree-sitter-cpp")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print("cần cài:", need or "không")
if need:
    sh("pip install -q " + " ".join(need))

import torch, transformers, datasets, sklearn, numpy
assert torch.cuda.is_available(), "Kaggle chưa gắn GPU — vào Settings → Accelerator → GPU"
print({"torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
       "datasets": datasets.__version__, "gpu": torch.cuda.get_device_name(0),
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 1)})
print("graphcodebert-base tải được?", bool(__import__("transformers", fromlist=["AutoModel"]).AutoModel.from_pretrained("microsoft/graphcodebert-base").config.hidden_size))


In [ ]:
# Rebuild dataset foundation POJ-104 ngay trên Kaggle.
# data/processed + data/splits bị .gitignore nên không có sẵn trong clone — phải build lại.
sh("python scripts/build_dataset.py --config configs/pairs/poj104.yaml")
py("import json; m = json.load(open('data/manifests/poj104.json')); "
   "print('leakage_passed =', m['leakage_passed']); "
   "print('pairs =', {k: v for k, v in m.get('pairs', {}).items()})")
sh("python scripts/env_report.py --output results/exp003_graphcodebert_poj104_512/env.json")
sh("df -h /kaggle/working | tail -1")

# Cùng sha256 manifest/pairs với EXP-002-512 là điều kiện để so sánh được.
sh("sha256sum data/manifests/poj104.json")


In [ ]:
# Warm-start 512: keo checkpoint 256 tu HuggingFace ve dung path config.
from huggingface_hub import hf_hub_download
HF_REPO = "Kichirou24/SemanticCodeCloneDetection"
HF_FILE = "m2_pair_256_weights_only.pt"
LOCAL = "results/exp003_graphcodebert_poj104/best_weights_only.pt"
p = hf_hub_download(HF_REPO, HF_FILE, local_dir=".", force_download=False, token=os.environ.get("HF_TOKEN"))
import shutil; shutil.copy(p, LOCAL)
print("warm-start weights OK:", LOCAL)


In [ ]:
# Preflight: GPU/VRAM/model/config trước khi tốn 8h. Phải IN ra báo cáo.
sh("python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert_512.yaml "
   "--mode preflight --device cuda")


In [ ]:
# Diagnostic: xác nhận model/gradient/param thật sự chạy trên GPU (gate của
# reports/010). Phải PASS trước khi train — và phải IN ra logit std > 0
# (nếu scores hội tụ về 0 thì classifier chưa học được, dừng lại).
sh("python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert_512.yaml "
   "--mode diagnostic --device cuda")


In [ ]:
# FULL TRAINING — cell này mất ~7,8h, đừng chạy Run All.
#
# Nếu session bị kill giữa chừng: chép lại last_checkpoint.pt từ version đã save,
# đổi MODE = "resume" rồi chạy lại cell này.
MODE = "train"  # "train" | "resume"
TIME_BUDGET_S = 9 * 3600  # dừng graceful trước giới hạn 12h của Kaggle
sh(f"python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert_512.yaml "
   f"--mode {MODE} --device cuda --time-budget-s {TIME_BUDGET_S}")


In [ ]:
# EVALUATE — chạy ngay trên Kaggle (~5 phút), KHÔNG tải checkpoint về máy.
#
# Vì sao cần cell này khi `train` đã tự evaluate: để artifact
# (metrics.json, predictions/*.jsonl, EVAL_COMPLETE.json) được sinh bởi MỘT lần
# chạy có thứ tự rõ ràng, và để chạy lại được sau khi tải xuống mà không tốn giờ local.
# Chạy lại nhiều lần là an toàn: ghi đè cùng một thư mục kết quả, không nhân bản.
sh("python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert_512.yaml "
   "--mode evaluate --device cuda")
OUT = "results/exp003_graphcodebert_poj104_512"
sh(f"test -f {OUT}/EVAL_COMPLETE.json && echo 'EVAL COMPLETE' || echo 'EVAL CHUA XONG'")
print(sh(f"cat {OUT}/metrics.json"))


In [ ]:
# Đóng gói kết quả để tải về — CHỈ metrics + predictions (vài MB), không kèm
# checkpoint: đã evaluate xong ở cell trên, và predictions/*.jsonl đủ để tính lại
# MỌI metric/threshold/error analysis mà không cần model (reports/011 §11.13).
OUT = "results/exp003_graphcodebert_poj104_512"
# Đổi sang True chỉ khi muốn giữ `best_weights_only.pt` (0,5GB, từ 1,44GB) để chạy
# lại evaluate ở máy khác. Mặc định False ⇒ tải về nhẹ nhất, và quota Kaggle đã trả
# cho 7,8h train thì không cần đốt thêm 5 phút GPU cho việc đã xong.
KEEP_WEIGHTS = False
if KEEP_WEIGHTS:
    py(f"from pathlib import Path; from src.experiments.exp003_graphcodebert import export_eval_weights; export_eval_weights(Path('{OUT}'))")

sh(f"tar --exclude='*checkpoint.pt' --exclude='*.tmp' -czf exp003_512_results.tar.gz {OUT}")
sh("ls -lh exp003_512_results.tar.gz")
sh(f"ls -lh {OUT}/")
print(sh(f"cat {OUT}/metrics.json"))


### Sau khi tải `exp003_512_results.tar.gz` về (vài MB)

1. Giải nén **đè** lên `results/exp003_graphcodebert_poj104_512/`.
2. **Không cần** chạy evaluate local — `metrics.json`, `threshold.json`,
   `confusion_matrix.json`, `model_stats.json` và `predictions/{validation,test}_scores.jsonl`
   đã có sẵn. Muốn tính lại số khác (F1 ở threshold khác, error analysis theo độ dài…)
   thì đọc `predictions/`, **không cần GPU**.
3. Nếu session Kaggle đã chết giữa chừng và bạn còn `last_checkpoint.pt` trong
   version đã save: chép vào thư mục result rồi `--mode resume`.
4. Cập nhật docs (repo yêu cầu code + docs tiến song):
   - `reports/012_graphcodebert_results.md` — kết quả test + so sánh M1 (DEC-016)
   - `docs/experiment_registry.md` — EXP-003-512 → COMPLETED
   - `configs/exp003_graphcodebert_512.yaml` — `experiment.status`

### Nhắc lại khi so sánh M1 vs M2
Cùng split, cùng pairs, cùng `max_length 256`, cùng hyperparameter, cùng runner,
cùng protocol threshold ⇒ khác biệt chỉ do pretrained weights. Nếu một bên đổi
hyperparameter giữa chừng thì kết luận RQ1 không còn đúng.